# Multimodal RAG System

In this lab, you'll build a complete RAG pipeline that retrieves relevant content from audio transcripts, slides, and video descriptions, then uses an LLM to generate grounded answers with citations.

Let's begin!

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download this notebook:</b> Click <em>"File"</em> → <em>"Download"</em>.</p>
</div>


## Connect to Snowflake

Load your credentials from the environment and create a Snowflake session.

In [ ]:
import os
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.core import Root, CreateMode
from snowflake.core.table import Table, TableColumn

_ = load_dotenv(override=True)

session = Session.builder.configs({
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "authenticator": "programmatic_access_token",
    "token": os.getenv("SNOWFLAKE_PAT"),
    "role": os.getenv("SNOWFLAKE_ROLE"),
    "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "database": os.getenv("SNOWFLAKE_DATABASE"),
}).create()
root = Root(session)

DATABASE = os.getenv("SNOWFLAKE_DATABASE")
WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE")
SHARED_SCHEMA = "shared"

current_user = session.get_current_user()
LEARNER_SCHEMA = current_user.strip('"').lower()

session.use_schema(LEARNER_SCHEMA)

print(f"Connected as {current_user}")
print(f"Using schema: {DATABASE}.{LEARNER_SCHEMA}")

## Create Unified Multimodal Content Table

To enable cross-modal search and retrieval, combine content from all three modalities — audio, slides, and video — into a single table with a shared schema. This lets you query once and rank results across all sources.

In [ ]:
unified_content_table = Table(
    name="multimodal_content",
    columns=[
        TableColumn(name="content_id", datatype="STRING"),
        TableColumn(name="source_type", datatype="STRING"),
        TableColumn(name="meeting_id", datatype="STRING"),
        TableColumn(name="meeting_part", datatype="STRING"),
        TableColumn(name="content_text", datatype="STRING"),
        TableColumn(name="source_reference", datatype="STRING"),
        TableColumn(name="text_embedding", datatype="VECTOR(FLOAT, 768)"),
    ]
)

root.databases[DATABASE].schemas[LEARNER_SCHEMA].tables.create(
    unified_content_table,
    mode=CreateMode.or_replace
)

print("Created multimodal_content table")

Populate the unified table by inserting content from audio transcript chunks, slide OCR text, and video descriptions — each tagged with its source type.

In [ ]:
session.sql("TRUNCATE TABLE IF EXISTS multimodal_content").collect()

session.sql(f"""
    INSERT INTO multimodal_content
    SELECT
        chunk_id AS content_id,
        'AUDIO' AS source_type,
        meeting_id,
        meeting_part,
        chunk_text AS content_text,
        CONCAT('Audio transcript chunk ', chunk_index, ' from meeting ', meeting_id, meeting_part) AS source_reference,
        text_embedding
    FROM {LEARNER_SCHEMA}.transcript_chunks
    WHERE text_embedding IS NOT NULL
""").collect()
print("Added audio transcript chunks")

session.sql(f"""
    INSERT INTO multimodal_content
    SELECT
        CONCAT(meeting_id, '_', meeting_part, '_', slide_filename) AS content_id,
        'SLIDE' AS source_type,
        meeting_id,
        meeting_part,
        extracted_text AS content_text,
        CONCAT('Slide ', slide_filename, ' from meeting ', meeting_id, meeting_part) AS source_reference,
        text_embedding
    FROM {LEARNER_SCHEMA}.slides_ocr
    WHERE text_embedding IS NOT NULL
""").collect()
print("Added slide OCR content")

session.sql(f"""
    INSERT INTO multimodal_content
    SELECT
        CONCAT(meeting_id, '_', meeting_part, '_video_', start_time) AS content_id,
        'VIDEO' AS source_type,
        meeting_id,
        meeting_part,
        description AS content_text,
        CONCAT('Video segment ', start_time, '-', end_time, ' from meeting ', meeting_id, meeting_part) AS source_reference,
        text_embedding
    FROM {SHARED_SCHEMA}.video_analysis
    WHERE text_embedding IS NOT NULL
""").collect()
print("Added video descriptions")


Check the content distribution across modalities to confirm all three sources were loaded.

In [ ]:
print("Unified Content Statistics:")
content_df = session.table("multimodal_content")
content_df.group_by("source_type").agg(
    F.count("*").alias("content_count"),
    F.avg(F.length(F.col("content_text"))).alias("avg_content_length")
).sort("source_type").show()

## Build RAG Retrieval Function

The retrieval step embeds the user's query and finds the most semantically similar content using vector cosine similarity. You can optionally filter by source type to focus on a specific modality.

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ &nbsp; <b>Each retrieval call takes a few seconds</b> as the query is embedded and compared against all content vectors.</p>

In [ ]:
def retrieve_context(query, top_k=5, source_filter=None):
    filter_clause = ""
    if source_filter:
        filter_clause = f"AND source_type = '{source_filter}'"

    results = session.sql(f"""
        WITH query_embedding AS (
            SELECT AI_EMBED(
                'snowflake-arctic-embed-m',
                '{query.replace("'", "''")}'
            ) AS embedding
        )
        SELECT
            content_id,
            source_type,
            meeting_id,
            meeting_part,
            content_text,
            source_reference,
            VECTOR_COSINE_SIMILARITY(text_embedding, q.embedding) AS similarity_score
        FROM multimodal_content m, query_embedding q
        WHERE text_embedding IS NOT NULL
        {filter_clause}
        ORDER BY similarity_score DESC
        LIMIT {top_k}
    """).collect()

    return results

Test the retrieval function with a sample query to see which content is returned and how it ranks.

In [ ]:
print("Test Retrieval: 'What was discussed about the budget?'")
test_results = retrieve_context("What was discussed about the budget?", top_k=5)

for row in test_results:
    print(f"\n[{row['SOURCE_TYPE']}] Score: {row['SIMILARITY_SCORE']:.4f}")
    print(f"  Reference: {row['SOURCE_REFERENCE']}")
    print(f"  Content: {row['CONTENT_TEXT'][:100]}...")

## Token Management

LLMs have context window limits. Use `AI_COUNT_TOKENS` to count tokens and ensure the retrieved context fits within the model's limits. This prevents truncation and keeps costs manageable.

In [ ]:
MAX_CONTEXT_TOKENS = 4000

def build_context_with_token_limit(results, max_tokens=MAX_CONTEXT_TOKENS):
    context_parts = []
    included_sources = []
    current_tokens = 0

    for row in results:
        content_piece = f"[{row['SOURCE_TYPE']}] {row['SOURCE_REFERENCE']}:\n{row['CONTENT_TEXT']}\n"

        token_count_result = session.sql(f"""
            SELECT AI_COUNT_TOKENS(
                'ai_complete',
                'claude-4-sonnet',
                '{content_piece.replace("'", "''")}'
            ) AS token_count
        """).collect()

        piece_tokens = token_count_result[0]['TOKEN_COUNT']

        if current_tokens + piece_tokens > max_tokens:
            print(f"  Stopping at {len(context_parts)} pieces ({current_tokens} tokens) — next piece would exceed limit")
            break

        context_parts.append(content_piece)
        included_sources.append(row['SOURCE_REFERENCE'])
        current_tokens += piece_tokens

    context_string = "\n---\n".join(context_parts)
    return context_string, included_sources, current_tokens

Test the token-limited context builder to see how many sources fit within the token budget.

In [ ]:
print("Building context with token limit...")
context, sources, tokens = build_context_with_token_limit(test_results)
print(f"Context built with {len(sources)} sources using {tokens} tokens")

## Generate Response with LLM

Use an LLM (`claude-4-sonnet` via `CORTEX.COMPLETE`) to answer the user's question based on the retrieved context. The prompt instructs the model to cite its sources so the user can verify the information.

<div style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px">
<p>🔍 &nbsp; <b>Results may vary:</b> LLM responses are non-deterministic. Your output may differ from what's shown in the video.</p>
</div>

In [ ]:
def generate_rag_response(query, context, sources):
    source_list = "\n".join([f"- {s}" for s in sources])

    prompt = f"""You are a helpful assistant answering questions about meeting content.
Use ONLY the provided context to answer the question. If the context doesn't contain
enough information to answer, say so.

Include citations in your response using the source references provided.
Format citations as [Source: reference].

CONTEXT:
{context}

AVAILABLE SOURCES FOR CITATION:
{source_list}

QUESTION: {query}

ANSWER:"""

    response = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'claude-4-sonnet',
            '{prompt.replace("'", "''")}'
        ) AS response
    """).collect()

    return response[0]['RESPONSE']

Test the full generation step — retrieve context, then generate a response with citations.

In [ ]:
print("Testing Full RAG Pipeline:")
test_query = "How did the team collaborate during presentations?"
print(f"Query: {test_query}\n")

response = generate_rag_response(test_query, context, sources)
print("Response:")
print(response)

## Complete RAG Pipeline

Tie together retrieval, token management, and generation into a single function. You can filter by source type to focus on a specific modality, or search across all of them.

In [ ]:
def multimodal_rag(query, top_k=10, source_filter=None, max_tokens=MAX_CONTEXT_TOKENS):
    results = retrieve_context(query, top_k=top_k, source_filter=source_filter)

    if not results:
        return {
            "response": "No relevant content found for your query.",
            "sources": [],
            "tokens_used": 0
        }

    context, sources, tokens = build_context_with_token_limit(results, max_tokens)
    response = generate_rag_response(query, context, sources)

    return {
        "response": response,
        "sources": sources,
        "tokens_used": tokens,
        "source_filter": source_filter
    }

## Run Queries

Try the RAG pipeline with different queries and source filters to see how it retrieves and synthesizes information across modalities.

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ &nbsp; <b>Each query takes 10–30 seconds</b> as it embeds the query, retrieves context, counts tokens, and generates a response.</p>

<div style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px">
<p>🔍 &nbsp; <b>Results may vary:</b> LLM responses are non-deterministic. Your output may differ from what's shown in the video.</p>
</div>

In [ ]:
queries = [
    ("How did the team decide on the project timeline and resource allocation?", None),
    ("What slides helped the team decide on the project?", "SLIDE"),
    ("How did the team reach consensus on next steps?", "VIDEO"),
]

for query, source_filter in queries:
    filter_label = f" (filtered to {source_filter})" if source_filter else " (all sources)"
    print(f"\nQuery: {query}{filter_label}")
    print("-" * 60)

    result = multimodal_rag(query, top_k=5, source_filter=source_filter)

    print(f"Tokens used: {result['tokens_used']}")
    print(f"Sources: {len(result['sources'])}")
    print(f"\nResponse:\n{result['response']}\n")

## Try Your Own Query

Enter your own question and optionally filter by source type to explore the meeting data.

In [ ]:
user_query = input("Enter your question (or press Enter to skip): ")

if user_query.strip():
    print("\nFilter options: AUDIO, SLIDE, VIDEO, or leave empty for all")
    filter_input = input("Enter source filter (or press Enter for all): ").strip().upper()
    source_filter = filter_input if filter_input in ["AUDIO", "SLIDE", "VIDEO"] else None

    print("\nProcessing your query...")
    result = multimodal_rag(user_query, top_k=10, source_filter=source_filter)

    print(f"\nTokens used: {result['tokens_used']}")
    print(f"Sources consulted: {len(result['sources'])}")
    for s in result['sources']:
        print(f"  - {s}")
    print(f"\nResponse:\n{result['response']}")

## Summary

In this lab, you:

1. **Unified three modalities** — Combined audio transcript chunks, slide OCR text, and video descriptions into a single `multimodal_content` table with a shared schema and `VECTOR(FLOAT, 768)` embeddings
2. **Built a retrieval function** — Embedded user queries and ranked results with `VECTOR_COSINE_SIMILARITY`, with optional filtering by source type
3. **Managed the context window** — Used `AI_COUNT_TOKENS` to pack retrieved passages into a token budget before sending them to the LLM
4. **Generated grounded answers** — Called `SNOWFLAKE.CORTEX.COMPLETE` with `claude-4-sonnet` to produce responses with inline source citations
5. **Assembled the full RAG pipeline** — Wrapped retrieval, token management, and generation into a single `multimodal_rag` function
6. **Ran cross-modal queries** — Asked questions across audio, slides, and video and inspected which sources the LLM used

You now have an end-to-end multimodal RAG system running entirely inside Snowflake — from raw audio, slides, and video in a stage, through ASR / OCR / VLM extraction and embeddings, to grounded answers with citations.


## References

- [VECTOR_COSINE_SIMILARITY](https://docs.snowflake.com/en/sql-reference/functions/vector_cosine_similarity) - SQL reference for the similarity function that drives retrieval in this lab
- [SNOWFLAKE.CORTEX.COMPLETE](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex) - Reference for the LLM generation function, including the list of supported models (such as `claude-4-sonnet`)
- [AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens) -  Reference for the token-counting helper used to keep retrieved context within the model's context window